# Social Media Addiction Analysis

This project looks at social media addiction using 5 datasets — covering user habits, platform usage, demographics, internet growth trends, and mental health in children. The analysis covers data loading, cleaning, EDA, visualization, correlation, regression, and hypothesis testing.

## Step 1 — Import Libraries and Load Data

Importing the required libraries and loading all 5 CSV files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Load all 5 datasets
smmh    = pd.read_csv('smmh.csv')
usage   = pd.read_csv('social_media_usage.csv')
cps     = pd.read_csv('cps_internet_2021.csv')
ntia    = pd.read_csv('ntia-analyze-table.csv')
child   = pd.read_csv('child23.csv')

print('All datasets loaded!')
print(f'  smmh    : {smmh.shape}')
print(f'  usage   : {usage.shape}')
print(f'  cps     : {cps.shape}')
print(f'  ntia    : {ntia.shape}')
print(f'  child   : {child.shape}')

## Step 2 — Data Cleaning (SMMH)

The SMMH dataset has long survey question headers, text-based responses, and some missing values. We rename the columns to shorter names, drop rows with nulls, fix the Age column type, and filter Gender to Male/Female only.

We also convert the `Daily_Hours` text categories to numeric values, and create two new columns: `Addiction_Score` (average of 3 addiction-related questions) and `Mental_Health_Score` (average of 3 mental health questions).

In [ ]:
# Rename columns to short clean names
smmh.columns = [
    'Timestamp', 'Age', 'Gender', 'Relationship_Status', 'Occupation',
    'Organization', 'Uses_SM', 'Platforms', 'Daily_Hours',
    'Purposeless_Use', 'Distracted', 'Restless',
    'Easily_Distracted', 'Bothered_by_Worries', 'Concentration_Issues',
    'Compare_Others', 'Feeling_Comparisons', 'Seek_Validation',
    'Feel_Depressed', 'Interest_Fluctuation', 'Sleep_Issues'
]

# Check missing values
print('Missing values:')
print(smmh.isnull().sum())

# Drop missing rows
smmh = smmh.dropna()

# Fix types
smmh['Age'] = pd.to_numeric(smmh['Age'], errors='coerce')
smmh = smmh.dropna(subset=['Age'])
smmh['Age'] = smmh['Age'].astype(int)

# Keep only Male / Female in Gender
smmh['Gender'] = smmh['Gender'].str.strip()
smmh = smmh[smmh['Gender'].isin(['Male', 'Female'])]

# Map Daily_Hours text → numeric
hours_map = {
    'Less than an Hour': 0.5,
    'Between 1 and 2 hours': 1.5,
    'Between 2 and 3 hours': 2.5,
    'Between 3 and 4 hours': 3.5,
    'Between 4 and 5 hours': 4.5,
    'More than 5 hours': 6.0
}
smmh['Daily_Hours_Num'] = smmh['Daily_Hours'].map(hours_map)

# Create Addiction Score = average of Q9, Q10, Q11 (scale 1–5)
smmh['Addiction_Score'] = smmh[['Purposeless_Use', 'Distracted', 'Restless']].mean(axis=1)

# Create Mental Health Score = average of depression + sleep + worry
smmh['Mental_Health_Score'] = smmh[['Feel_Depressed', 'Sleep_Issues', 'Bothered_by_Worries']].mean(axis=1)

print(f'\nClean dataset: {smmh.shape[0]} rows, {smmh.shape[1]} columns')

## Step 2b — Addiction Indicator: Late-Night Usage

The `Timestamp` column records when each survey response was submitted. People filling out a social media survey between **10 PM and 4 AM** are likely using their phones late at night — a key behavioral indicator of social media addiction.

We extract the hour from the timestamp, create a `Late_Night_User` flag, and compare addiction scores between late-night and normal users.

In [ ]:
# Parse Timestamp and extract hour
smmh['Timestamp'] = pd.to_datetime(smmh['Timestamp'])
smmh['Hour'] = smmh['Timestamp'].dt.hour

# Flag late-night users: 10 PM (22) to 4 AM (3)
smmh['Late_Night_User'] = smmh['Hour'].apply(lambda h: 1 if (h >= 22 or h <= 3) else 0)

# Count of late-night vs normal users
counts = smmh['Late_Night_User'].value_counts().rename({0: 'Normal Hours', 1: 'Late Night (10PM–4AM)'})
print('=== Late-Night Usage Count ===')
print(counts)
print(f'\nLate-night users: {counts.get("Late Night (10PM–4AM)", 0)} ({counts.get("Late Night (10PM–4AM)", 0)/len(smmh)*100:.1f}%)')

# Bar chart — Late-night vs Normal user count
plt.figure(figsize=(6, 4))
plt.bar(counts.index, counts.values, color=['steelblue', 'salmon'], edgecolor='black')
plt.title('Late-Night vs Normal Hour Social Media Users')
plt.xlabel('User Type')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Compare Addiction Score: Late-night vs Normal
avg_addiction = smmh.groupby('Late_Night_User')['Addiction_Score'].mean().rename({0: 'Normal Hours', 1: 'Late Night'}).round(2)
print('\n=== Avg Addiction Score by Usage Timing ===')
print(avg_addiction)

# Bar chart — Addiction Score comparison
plt.figure(figsize=(6, 4))
plt.bar(avg_addiction.index, avg_addiction.values, color=['steelblue', 'salmon'], edgecolor='black')
plt.title('Avg Addiction Score: Late-Night vs Normal Hour Users')
plt.xlabel('User Type')
plt.ylabel('Avg Addiction Score (1–5)')
plt.ylim(0, 5)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('Interpretation: Late-night users tend to have a higher addiction score,')
print('confirming that time-of-access is a meaningful addiction indicator.')

## Step 3 — Descriptive Statistics

First we use `.describe()` to get an overview of all key numeric columns. Then we print individual stats for `Addiction_Score` manually to understand what each measure actually means.

In [ ]:
# Summary stats for key columns
cols = ['Age', 'Daily_Hours_Num', 'Addiction_Score', 'Mental_Health_Score', 'Feel_Depressed', 'Sleep_Issues']
print('=== Descriptive Statistics ===')
print(smmh[cols].describe().round(2))

In [ ]:
# Manual stats — explain each formula for viva
score = smmh['Addiction_Score']
print('=== Manual Stats: Addiction Score ===')
print(f'  Mean   (avg value)         : {score.mean():.2f}')
print(f'  Median (middle value)      : {score.median():.2f}')
print(f'  Std Dev (spread)           : {score.std():.2f}')
print(f'  Min                        : {score.min():.2f}')
print(f'  Max                        : {score.max():.2f}')
print()
print('=== Gender Count ===')
print(smmh['Gender'].value_counts())
print()
print('=== Occupation Count ===')
print(smmh['Occupation'].value_counts())

## Step 4 — EDA: Who Is Most Addicted?

We use `groupby()` to find the average addiction score broken down by gender, occupation, and relationship status. This helps identify which demographic groups show higher addiction patterns.

In [ ]:
# Addiction Score by Gender
print('=== Avg Addiction Score by Gender ===')
print(smmh.groupby('Gender')['Addiction_Score'].mean().round(2))

print()
# Addiction Score by Occupation
print('=== Avg Addiction Score by Occupation ===')
print(smmh.groupby('Occupation')['Addiction_Score'].mean().round(2).sort_values(ascending=False))

print()
# Addiction Score by Relationship Status
print('=== Avg Addiction Score by Relationship Status ===')
print(smmh.groupby('Relationship_Status')['Addiction_Score'].mean().round(2).sort_values(ascending=False))

In [ ]:
# Outlier Detection using IQR
print('=== Outlier Detection: Addiction Score (IQR Method) ===')
Q1  = smmh['Addiction_Score'].quantile(0.25)
Q3  = smmh['Addiction_Score'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = smmh[(smmh['Addiction_Score'] < lower) | (smmh['Addiction_Score'] > upper)]
print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'  Lower bound={lower:.2f}, Upper bound={upper:.2f}')
print(f'  Outliers found: {len(outliers)}')

## Step 5 — Which Platform Is Most Addictive?

Using `social_media_usage.csv` to compare platforms by how much time users spend on them, and how often they post and receive likes. High engagement (posts + likes) is a sign of addictive usage patterns.

In [ ]:
# Preview
print(usage.head())
print(f'\nShape: {usage.shape}')
print(f'Apps: {usage["App"].unique()}')

In [ ]:
# Average daily minutes by platform
app_avg = usage.groupby('App')['Daily_Minutes_Spent'].mean().sort_values(ascending=False).round(1)
print('=== Avg Daily Minutes by App ===')
print(app_avg)

# Bar Chart
plt.figure(figsize=(8, 4))
plt.bar(app_avg.index, app_avg.values, color='steelblue', edgecolor='black')
plt.title('Average Daily Minutes Spent by Social Media Platform')
plt.xlabel('Platform')
plt.ylabel('Avg Minutes/Day')
plt.xticks(rotation=20)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Seaborn Boxplot — spread of time per platform
plt.figure(figsize=(9, 4))
sns.boxplot(data=usage, x='App', y='Daily_Minutes_Spent', palette='Blues')
plt.title('Distribution of Daily Minutes Spent per Platform (Seaborn Boxplot)')
plt.xlabel('Platform')
plt.ylabel('Daily Minutes')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print('Interpretation: A wider box = more variation in usage time')

In [ ]:
# Avg Posts & Likes per App — engagement proxy for addiction
eng = usage.groupby('App')[['Posts_Per_Day', 'Likes_Per_Day']].mean().round(1)
print('=== Avg Engagement by App ===')
print(eng.sort_values('Likes_Per_Day', ascending=False))

# Grouped bar chart
x = np.arange(len(eng.index))
width = 0.35

plt.figure(figsize=(9, 4))
plt.bar(x - width/2, eng['Posts_Per_Day'], width, label='Posts/Day', color='steelblue')
plt.bar(x + width/2, eng['Likes_Per_Day'] / 10, width, label='Likes/Day (÷10)', color='salmon')
plt.xticks(x, eng.index, rotation=20)
plt.title('Avg Posts & Likes per Day by Platform')
plt.xlabel('Platform')
plt.ylabel('Count')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Step 6 — Digital Dependency by Demographics

Using the CPS 2021 dataset to see how digital assistant usage (a proxy for digital dependency) varies across education levels and age groups.

In [ ]:
print(cps.head(3))
print(f'Shape: {cps.shape}')

In [ ]:
# Digital Assistant usage rate by Education Level
# Uses_Digital_Assistant: 1=Yes, 2=No
cps['Uses_DA'] = (cps['Uses_Digital_Assistant'] == 1).astype(int)

ed_da = cps.groupby('Education_Label')['Uses_DA'].mean().sort_values(ascending=False).round(3) * 100
ed_da = ed_da[ed_da.index.isin([
    'High school diploma / GED', 'Some college, no degree',
    "Associate degree (academic)", "Bachelor's degree",
    "Master's degree", "Doctorate degree"
])]

plt.figure(figsize=(8, 4))
plt.barh(ed_da.index, ed_da.values, color='steelblue', edgecolor='black')
plt.title('% Using Digital Assistants by Education Level')
plt.xlabel('Usage Rate (%)')
plt.tight_layout()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()
print('Higher education → higher digital assistant dependency')

In [ ]:
# Digital dependency by Age group
bins  = [0, 18, 25, 35, 50, 65, 100]
labels= ['<18', '18-25', '26-35', '36-50', '51-65', '65+']
cps['Age_Group'] = pd.cut(cps['Age'], bins=bins, labels=labels)

age_da = cps.groupby('Age_Group', observed=True)['Uses_DA'].mean().round(3) * 100

plt.figure(figsize=(7, 4))
plt.bar(age_da.index.astype(str), age_da.values, color='salmon', edgecolor='black')
plt.title('Digital Assistant Usage Rate by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Usage Rate (%)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print('Younger age groups tend to use digital tools more actively')

## Step 7 — Internet Growth Over Time

Using NTIA data to plot how the percentage of internet users in the US has changed from 1998 to 2023. This gives context to how widespread digital dependency has become over the decades.

In [ ]:
# Filter internet users proportion over years
trend = ntia[ntia['variable'] == 'internetUser'][['dataset', 'usProp']].dropna()

# Keep one row per year (deduplicate by taking mean)
trend['Year'] = trend['dataset'].str.extract(r'(\d{4})').astype(int)
trend = trend.groupby('Year')['usProp'].mean().reset_index()
trend['usProp_pct'] = (trend['usProp'] * 100).round(1)

print(trend.to_string(index=False))

In [ ]:
# Line Plot — Internet User Growth (addiction proxy)
plt.figure(figsize=(9, 4))
plt.plot(trend['Year'], trend['usProp_pct'], marker='o', color='steelblue', linewidth=2)
for _, row in trend.iterrows():
    plt.text(row['Year'], row['usProp_pct'] + 0.8, f"{row['usProp_pct']}%", ha='center', fontsize=7)
plt.title('US Internet Usage Growth (1998–2023) — NTIA Data')
plt.xlabel('Year')
plt.ylabel('% of Population Online')
plt.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print('Interpretation: Internet usage has grown from ~35% in 1998 to ~85% in 2023')
print('This steady growth reflects increasing digital dependency over decades.')

## Step 8 — Anxiety and Depression in Children

The `child23.csv` dataset has survey responses from children aged 0–17. We look at how often they experience anxiety and depression across different age groups.

Note: the scale is 1 = Daily, 5 = Never — so a lower score means more frequent distress.

In [ ]:
# Select relevant columns: Age, Anxiety, Depression, Focus difficulty
ch = child[['AGEP_C', 'ANXFREQ_C', 'DEPFREQ_C', 'BEHDFCNTR_C']].copy()
ch.columns = ['Age', 'Anxiety_Freq', 'Depression_Freq', 'Attention_Difficulty']

# Keep valid values only (1–5 scale; 7/9 are NA codes)
for col in ['Anxiety_Freq', 'Depression_Freq', 'Attention_Difficulty']:
    ch = ch[ch[col].isin([1, 2, 3, 4, 5])]

ch = ch.dropna()
print(f'Clean child dataset: {ch.shape}')
print(ch.describe().round(2))

In [ ]:
# Create Age Groups for children
ch['Age_Group'] = pd.cut(ch['Age'], bins=[0, 5, 9, 13, 17], labels=['0-5', '6-9', '10-13', '14-17'])

# Average Anxiety & Depression by Age Group
age_mental = ch.groupby('Age_Group', observed=True)[['Anxiety_Freq', 'Depression_Freq']].mean().round(2)
print('=== Avg Anxiety & Depression by Child Age Group ===')
print(age_mental)

# Bar chart — Anxiety by age group
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(age_mental.index.astype(str), age_mental['Anxiety_Freq'], color='salmon', edgecolor='black')
axes[0].set_title('Avg Anxiety Frequency by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Avg Score (1=Most, 5=Never)')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

axes[1].bar(age_mental.index.astype(str), age_mental['Depression_Freq'], color='steelblue', edgecolor='black')
axes[1].set_title('Avg Depression Frequency by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Avg Score (1=Most, 5=Never)')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('Mental Health Indicators in Children (child23.csv)', fontsize=12)
plt.tight_layout()
plt.show()
print('Note: Score 1 = Daily, 5 = Never. Lower bars = more frequent distress.')

## Step 9 — Correlation Analysis

Computing the correlation between daily hours, addiction score, mental health score, and related columns from SMMH. Values close to 1 mean the two variables tend to increase together; values close to -1 mean the opposite.

In [ ]:
corr_cols = ['Daily_Hours_Num', 'Addiction_Score', 'Mental_Health_Score',
             'Feel_Depressed', 'Sleep_Issues', 'Concentration_Issues', 'Seek_Validation']

corr_matrix = smmh[corr_cols].corr().round(2)
print('=== Correlation Matrix ===')
print(corr_matrix)

In [ ]:
# Seaborn heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap — Social Media Addiction & Mental Health')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  +1.0 = strong positive relationship')
print('   0.0 = no relationship')
print('  -1.0 = strong negative relationship')

## Step 10 — Linear Regression

We want to see if daily screen time predicts addiction score. Instead of using scikit-learn, the slope and intercept are calculated manually using the standard regression formulas:

- `m = Σ((x - x̄)(y - ȳ)) / Σ((x - x̄)²)`
- `b = ȳ - m * x̄`

In [ ]:
reg = smmh[['Daily_Hours_Num', 'Addiction_Score']].dropna()
x = reg['Daily_Hours_Num']
y = reg['Addiction_Score']

# Manual slope & intercept
x_mean = x.mean()
y_mean = y.mean()
m = ((x - x_mean) * (y - y_mean)).sum() / ((x - x_mean)**2).sum()
b = y_mean - m * x_mean

print(f'Regression: Addiction_Score = {m:.4f} × Daily_Hours + {b:.4f}')
print(f'Slope     : {m:.4f}  → for every 1 extra hour, addiction increases by {m:.4f}')
print(f'Intercept : {b:.4f}')

In [ ]:
# Scatter + regression line
y_pred = m * x + b

plt.figure(figsize=(7, 4))
plt.scatter(x, y, alpha=0.4, color='steelblue', label='Actual Data')
plt.plot(sorted(x), [m * xi + b for xi in sorted(x)], color='red', linewidth=2, label=f'y = {m:.3f}x + {b:.3f}')
plt.title('Linear Regression: Daily Hours vs Addiction Score')
plt.xlabel('Daily Hours on Social Media')
plt.ylabel('Addiction Score')
plt.legend()
plt.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Step 11 — Hypothesis Testing (t-test)

Testing whether males and females have a significantly different addiction score.

- **H0:** The mean addiction score is the same for both genders
- **H1:** There is a significant difference

The t-statistic is calculated manually using the two-sample t-test formula. If `|t| > 1.96`, we reject H0 at the 5% significance level.

In [ ]:
males   = smmh[smmh['Gender'] == 'Male']['Addiction_Score'].dropna()
females = smmh[smmh['Gender'] == 'Female']['Addiction_Score'].dropna()

n1, mean1, std1 = len(males),   males.mean(),   males.std()
n2, mean2, std2 = len(females), females.mean(), females.std()

# t = (mean1 - mean2) / sqrt(std1²/n1 + std2²/n2)
t_stat = (mean1 - mean2) / math.sqrt((std1**2 / n1) + (std2**2 / n2))

print(f'Male   — n={n1}, mean={mean1:.2f}, std={std1:.2f}')
print(f'Female — n={n2}, mean={mean2:.2f}, std={std2:.2f}')
print()
print(f't-statistic     : {t_stat:.4f}')
print(f'Critical value  : ±1.96  (α=0.05, two-tailed)')
print()
if abs(t_stat) > 1.96:
    print('Result  : |t| > 1.96 → REJECT H0')
    print('Conclusion: Significant difference in addiction between males and females.')
else:
    print('Result  : |t| ≤ 1.96 → FAIL TO REJECT H0')
    print('Conclusion: No significant difference in addiction between males and females.')

In [ ]:
# Overlapping histograms — Male vs Female addiction
plt.figure(figsize=(7, 4))
plt.hist(males,   bins=10, alpha=0.6, color='steelblue', label='Male',   edgecolor='black')
plt.hist(females, bins=10, alpha=0.6, color='salmon',    label='Female', edgecolor='black')
plt.axvline(mean1, color='blue', linestyle='--', linewidth=1.5, label=f'Male mean = {mean1:.2f}')
plt.axvline(mean2, color='red',  linestyle='--', linewidth=1.5, label=f'Female mean = {mean2:.2f}')
plt.title('Addiction Score Distribution: Male vs Female')
plt.xlabel('Addiction Score')
plt.ylabel('Frequency')
plt.legend()
plt.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Final Summary

Printing a consolidated summary of the project — datasets used, steps completed, and the key numeric results.

In [ ]:
print('=' * 60)
print('  PROJECT: Social Media Addiction Analysis')
print('=' * 60)
print()
print('  Datasets Used:')
print(f'    smmh.csv              : {smmh.shape[0]} rows (main survey)')
print(f'    social_media_usage    : {usage.shape[0]} rows (platform data)')
print(f'    cps_internet_2021     : {cps.shape[0]} rows (US demographics)')
print(f'    ntia-analyze-table    : {ntia.shape[0]} rows (trend data 1998-2023)')
print(f'    child23               : {child.shape[0]} rows (children mental health)')
print()
print('  Steps Completed:')
print('    Step 1  : Load all 5 datasets                    (Unit I)')
print('    Step 2  : Data cleaning — SMMH                   (Unit II)')
print('    Step 2b : Late-night usage addiction indicator    (Unit III)')
print('    Step 3  : Descriptive statistics                  (Unit V)')
print('    Step 4  : EDA — groupby, outlier detection        (Unit IV)')
print('    Step 5  : Platform addiction — usage.csv          (Unit III+IV)')
print('    Step 6  : Digital dependency — CPS demographics   (Unit III+IV)')
print('    Step 7  : Internet growth trend — NTIA line plot  (Unit III)')
print('    Step 8  : Children anxiety/depression — child23   (Unit II+IV)')
print('    Step 9  : Correlation heatmap — SMMH              (Unit IV)')
print('    Step 10 : Manual linear regression (y=mx+b)       (Unit V)')
print('    Step 11 : Manual t-test (no scipy)                (Unit V)')
print()
print('  Key Findings:')
print(f'    Avg Addiction Score        : {smmh["Addiction_Score"].mean():.2f} / 5')
print(f'    Avg Daily Hours (SM)       : {smmh["Daily_Hours_Num"].mean():.2f} hrs')
print(f'    Regression Slope           : {m:.4f} (more hours → more addiction)')
print(f'    t-statistic (M vs F)       : {t_stat:.4f}')
print(f'    Internet users in 2023     : ~85% of US population')
print('=' * 60)